# 001 · Logging Basics

> **Standard Library › Logging & Config** · Python 3.14

The `logging` module is Python's built-in, framework-agnostic logging API — the role `Microsoft.Extensions.Logging` (`ILogger<T>`) plays in .NET. Libraries write to named loggers; the application decides where the messages go and which levels are shown.

**In this notebook you will learn:**

- how `logging` maps onto `ILogger`, log levels and categories
- `getLogger(__name__)` and the dotted logger hierarchy
- `basicConfig` — and why it can silently do nothing (especially in Jupyter)
- lazy `%`-style arguments vs f-strings
- `logger.exception` for logging errors with tracebacks
- the rules for logging *inside a library* (no root logger, `NullHandler`)

## .NET mental model

| Microsoft.Extensions.Logging | Python `logging` |
|---|---|
| `ILogger<OrderService>` (category = type name) | `logging.getLogger(__name__)` (name = module path) |
| `ILoggerFactory.CreateLogger("Shop.Orders")` | `logging.getLogger("shop.orders")` |
| `builder.Logging.AddConsole()` + `SetMinimumLevel` | `logging.basicConfig(level=..., stream=...)` |
| logging provider (`ConsoleLoggerProvider`, …) | **handler** (`StreamHandler`, `FileHandler`, …) |
| `logger.LogInformation("Order {Id}", id)` | `logger.info("Order %s", order_id)` |
| `logger.LogError(ex, "Failed")` | `logger.exception("Failed")` inside `except` |

**Level mapping** — Python has no `Trace`, and its numbers are spaced by 10 so you can define levels in between:

| .NET `LogLevel` | Python level | Value |
|---|---|---|
| `Trace` (0) | — (use `DEBUG`, or a custom level such as 5) | — |
| `Debug` (1) | `DEBUG` | 10 |
| `Information` (2) | `INFO` | 20 |
| `Warning` (3) | `WARNING` (the default threshold) | 30 |
| `Error` (4) | `ERROR` | 40 |
| `Critical` (5) | `CRITICAL` | 50 |
| `None` (6) | (disable with `logging.disable(...)` or a level above 50) | — |

In [ ]:
import logging

for name in ["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"]:
    print(f"{name:<9}", getattr(logging, name))
print(logging.getLevelName(30))          # 'WARNING'

## Out of the box: only warnings and above

Without any configuration, the **root logger** has level `WARNING` and no handlers. Python then falls back to a built-in "last resort" handler that prints bare messages to **stderr** — so `info`/`debug` are silently dropped.

In this notebook we want log output to appear inline like `print`, so every handler we configure writes to `sys.stdout`. (In a real console app, logging to the default **stderr** is usually what you want — it keeps logs separate from program output.)

In [ ]:
root = logging.getLogger()
print(root.name, logging.getLevelName(root.level), root.handlers)   # root WARNING []

## `basicConfig` — one-line setup

`logging.basicConfig(...)` attaches a handler with a formatter to the **root logger**. It's the equivalent of `builder.Logging.ClearProviders().AddSimpleConsole()` with a minimum level.

```csharp
// C#
using var factory = LoggerFactory.Create(b => b
    .AddSimpleConsole()
    .SetMinimumLevel(LogLevel.Debug));
ILogger logger = factory.CreateLogger("demo");
logger.LogInformation("Application started");
```

In [ ]:
import sys

logging.basicConfig(
    level=logging.DEBUG,
    format="%(levelname)-8s %(name)s: %(message)s",
    stream=sys.stdout,        # show inline in the notebook (default would be stderr)
)

log = logging.getLogger("demo")
log.debug("debug details")
log.info("Application started")
log.warning("Disk space low")
log.error("Could not save order")
log.critical("Database unreachable")

⚠️ **Gotcha:** `basicConfig` **does nothing if the root logger already has handlers**. In a script that's fine (you call it once in `main`). In Jupyter it bites constantly: re-running the cell, or a library/IDE that already configured logging, makes your new settings silently ignored. Pass `force=True` (3.8+) to remove existing root handlers first.

In [ ]:
logging.basicConfig(level=logging.INFO, format="NEW FORMAT | %(message)s", stream=sys.stdout)
log.info("still using the old format — the call above was ignored")

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s", stream=sys.stdout, force=True)
log.info("force=True replaced the handler")
log.debug("and DEBUG is now filtered out")

## `getLogger(__name__)` and the logger hierarchy

Loggers are **singletons by name**: `getLogger("shop")` always returns the same object. Dots form a hierarchy — `shop.orders` is a child of `shop`, which is a child of the root logger. Records **propagate** up to ancestors' handlers, and a logger without its own level inherits its parent's *effective level*.

Convention: in each module write `logger = logging.getLogger(__name__)`. `__name__` is the module's dotted import path (e.g. `shop.orders`), so you get a hierarchy that mirrors your package for free — just like `ILogger<T>` uses the full type name as the category.

In [ ]:
shop = logging.getLogger("shop")
orders = logging.getLogger("shop.orders")

print(logging.getLogger("shop.orders") is orders)    # True — same object
print(orders.parent is shop, shop.parent is root)    # True True
print(__name__)                                      # '__main__' in a notebook or script

shop.setLevel(logging.WARNING)                        # quieten the whole "shop" subtree
print(logging.getLevelName(orders.getEffectiveLevel()))   # WARNING — inherited
orders.info("filtered: shop is WARNING")
orders.warning("Order 1042 is delayed")

This is how you tune categories, like `"Logging:LogLevel:Microsoft.EntityFrameworkCore": "Warning"` in `appsettings.json`: set levels on parent loggers. A child can still opt back in:

In [ ]:
logging.getLogger("shop.payments").setLevel(logging.DEBUG)
logging.getLogger("shop.payments").debug("payments debug is on")   # shown
logging.getLogger("shop.inventory").debug("inventory debug")        # hidden (inherits WARNING)
shop.setLevel(logging.NOTSET)                                       # back to inheriting from root

## Lazy `%`-style arguments vs f-strings

```csharp
// C# — message template + args; formatting is deferred and args are captured
logger.LogInformation("Shipped order {OrderId} to {City}", orderId, city);
```

Python's logging uses old-style `%` placeholders with the arguments passed separately. The string is only formatted **if the record is actually emitted**.

In [ ]:
order_id, city = 1042, "Springfield"
log.info("Shipped order %s to %s", order_id, city)            # preferred
log.info("Shipped order %(id)s to %(city)s", {"id": order_id, "city": city})   # named, via one dict

Why not f-strings? An f-string is evaluated **before** `log.debug` is even called, so you pay for formatting (and for any expensive `__str__`/`__repr__`) even when debug logging is off. Lazy args also keep the *template* (`record.msg`) separate from the values (`record.args`), which structured-logging handlers can use — similar to message templates in `ILogger`.

In [ ]:
class Expensive:
    calls = 0
    def __str__(self) -> str:
        Expensive.calls += 1
        return "expensive!"

payload = Expensive()
log.debug(f"payload={payload}")       # DEBUG is off, but the f-string already ran __str__
log.debug("payload=%s", payload)      # DEBUG is off -> never formatted
print("__str__ calls:", Expensive.calls)   # 1

**Tip:** for truly expensive values (not just formatting), guard with `if log.isEnabledFor(logging.DEBUG):` — the counterpart of `if (logger.IsEnabled(LogLevel.Debug))`.

⚠️ **Gotcha:** the placeholders are `%s`/`%d`/`%r` — **not** `{}` and not `{Name}`. Mismatched arguments don't raise in your code; logging prints a "--- Logging error ---" report to stderr instead and carries on. Linters such as Ruff/Pylint flag f-strings and bad placeholders in logging calls.

## Logging exceptions

Inside an `except` block, `logger.exception(msg)` logs at `ERROR` level **and appends the traceback** — like `logger.LogError(ex, "...")`. For other levels pass `exc_info=True`.

In [ ]:
def parse_quantity(text: str) -> int:
    return int(text)

try:
    parse_quantity("three")
except ValueError:
    log.exception("Could not parse quantity for order %s", order_id)

In [ ]:
try:
    {}["missing"]
except KeyError:
    log.warning("Optional key missing, using default", exc_info=True)

(The file paths in these tracebacks point at the temporary files Jupyter creates for each cell, so they differ per machine.)

⚠️ **Gotcha:** don't write `log.error(f"Failed: {e}")` and lose the stack trace. `log.exception(...)` (or `exc_info=True`) captures the full traceback at no extra effort.

## Logging in libraries: don't touch the root logger

Rules of thumb (the same spirit as "libraries depend on `ILogger`, apps configure providers"):

1. **Libraries** create `logger = logging.getLogger(__name__)` and just log. They never call `basicConfig`, never add console/file handlers, and never log via the module-level `logging.info(...)` functions (those use the **root** logger — and implicitly call `basicConfig()` if root has no handlers!).
2. A library may add a `logging.NullHandler()` to its top-level logger. It does nothing, but it means "this library's logging is handled", so Python's last-resort stderr output doesn't kick in when the application hasn't configured logging.
3. **Applications** configure logging once, at start-up.

In [ ]:
# --- inside a hypothetical library package "acme_utils" ---
lib_logger = logging.getLogger("acme_utils")
lib_logger.addHandler(logging.NullHandler())

def slugify(text: str) -> str:
    lib_logger.debug("slugifying %r", text)
    return "-".join(text.lower().split())

# --- in the application ---
logging.getLogger("acme_utils").setLevel(logging.DEBUG)   # app opts in to the library's debug logs
print(slugify("Hello Big World"))

## Turning logging off temporarily

`logging.disable(level)` suppresses everything at or below `level` globally — handy in tests. `logging.disable(logging.NOTSET)` turns it back on.

In [ ]:
logging.disable(logging.CRITICAL)
log.critical("you will not see this")
logging.disable(logging.NOTSET)
log.info("logging is back")

## Cleaning up

Remove the handlers we added so later cells (or notebooks sharing the kernel) start fresh.

In [ ]:
for h in root.handlers[:]:
    root.removeHandler(h)
    h.close()
for name in ["shop.payments", "acme_utils"]:
    logging.getLogger(name).setLevel(logging.NOTSET)
root.setLevel(logging.WARNING)
print(root.handlers, logging.getLevelName(root.level))

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Get a logger | `ILogger<T>` via DI | `logger = logging.getLogger(__name__)` |
| Console + min level | `AddConsole()` + `SetMinimumLevel(...)` | `logging.basicConfig(level=..., stream=...)` |
| Re-configure | rebuild the factory | `basicConfig(..., force=True)` |
| Information | `LogInformation("Order {Id}", id)` | `logger.info("Order %s", order_id)` |
| Log exception | `LogError(ex, "msg")` | `logger.exception("msg")` in `except` |
| Check level | `logger.IsEnabled(LogLevel.Debug)` | `logger.isEnabledFor(logging.DEBUG)` |
| Category filter | `"LogLevel": { "Shop": "Warning" }` | `logging.getLogger("shop").setLevel(logging.WARNING)` |
| Library default | depend on `ILogger` only | `getLogger(__name__)` + `NullHandler()` |

More pipeline control (handlers, formatters, filters, `dictConfig`) is in *Standard Library › Logging & Config › 002 · Configuring Logging*.

## Exercises

1. Configure logging (with `force=True`, to `sys.stdout`) so that messages look like `INFO | inventory | Restocked 5 units of apple`. Log that message with lazy arguments from a logger named `inventory`.
2. Create loggers `app`, `app.db` and `app.web`. Set `app` to `WARNING` and `app.db` to `DEBUG`. Which of these appear: `app.db.debug(...)`, `app.web.info(...)`, `app.web.error(...)`? Verify by running.
3. Write a function `safe_divide(a, b)` that returns `None` and logs the exception with traceback when `b == 0`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import logging
import sys

# 1 — format string with levelname and name
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s",
                    stream=sys.stdout, force=True)
logging.getLogger("inventory").info("Restocked %d units of %s", 5, "apple")

# 2 — app.db.debug: shown (own level DEBUG)
#     app.web.info: hidden (inherits WARNING from app)
#     app.web.error: shown
logging.getLogger("app").setLevel(logging.WARNING)
logging.getLogger("app.db").setLevel(logging.DEBUG)
logging.getLogger("app.db").debug("db debug")
logging.getLogger("app.web").info("web info")
logging.getLogger("app.web").error("web error")

# 3 — logger.exception inside except
logger = logging.getLogger("calc")

def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        logger.exception("Division failed: %s / %s", a, b)
        return None

print(safe_divide(10, 2), safe_divide(1, 0))
```

</details>